# Compare beam trackers: `SimplePXRLoader` vs `PXRLoader`

Point this at a folder of `.fits` files, tweak the parameters, and process the
same data with **both** trackers so you can see where they agree and disagree.

- **simple** — the new median-filter + local-argmax tracker
  (`SimplePXRLoader`), with a per-scan seed and a re-seed at the `sam_z` move.
- **current** — the existing SNR-gated tracker (`PXRLoader.process`).

Both use the *same* reduction config; the only difference is how the beam
centre is found. Run the cells top to bottom.

## 1. Parameters — edit these

In [81]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pxr_reduce.config import ReductionConfig
from pxr_reduce.core import PXRLoader
from pxr_reduce.simple_track import SimplePXRLoader

# Show pxr-reduce INFO logs (frame shape, per-stage timing summary) inline.
logging.basicConfig(level=logging.WARNING, format="%(message)s")
logging.getLogger("pxr_reduce").setLevel(logging.INFO)

# ---- Point this at your data -------------------------------------------
DATA_PATH = r"C:/path/to/your/data"   # folder containing .fits files
FILE_GLOB = "*.fits"

# ---- Simple-tracker knobs ----------------------------------------------
SEARCH_RADIUS = 15    # px the beam may drift between sequential frames
FILTER_SIZE   = 5     # median-filter kernel used to FIND the beam

# ---- Shared reduction config (both trackers use these) -----------------
CONFIG_KW = dict(
    detector="default",
    trim_x=20,
    trim_y=20,
    roi_height=40,
    roi_width=40,
    dark_pix_offset=50,
    dezinger=True,
)

## 2. Find the files

In [82]:
files = sorted(Path(DATA_PATH).glob(FILE_GLOB))
print(f"Found {len(files)} file(s) in {DATA_PATH}")
assert files, "No FITS files found - check DATA_PATH and FILE_GLOB."

Found 0 file(s) in C:/path/to/your/data


AssertionError: No FITS files found - check DATA_PATH and FILE_GLOB.

## 3. Process with the **simple** tracker

`search_radius` and `filter_size` override the config defaults here.

A live progress bar shows the current **scan**, frame **idx**, **beam** position
and **step** (`seed` vs `track`). Watch it to see the per-frame rate and ETA:

- If it looks stuck, the logged `First frame loaded: trimmed shape ...` line
  tells you how large each frame is (big frames = slow median filter).
- If it stops advancing on one index, that frame is the one hanging.
- The end-of-run line breaks time down by stage
  (`load / locate / dezinger / integrate`) so you can see what dominates.
- Set `verbose=True` to log those timings for **every** frame.

In [ ]:
simple = SimplePXRLoader(files, ReductionConfig(**CONFIG_KW))
simple.process(
    search_radius=SEARCH_RADIUS,
    filter_size=FILTER_SIZE,
    progress=True,   # live per-frame bar
    verbose=False,   # set True for per-frame stage timings
)
simple.data[["fits_index", "scan", "sam_th", "beam_spot", "counts_refl"]].head()

## 4. Process with the **current** tracker

In [ ]:
current = PXRLoader(files, ReductionConfig(**CONFIG_KW))
current.process()
current.data[["fits_index", "scan", "sam_th", "beam_spot", "beam_found", "beam_snr"]].head()

## 5. Compare beam positions

`dist` is the pixel distance between the two trackers' beam centres for each
frame. Large values are the frames where they disagree.

In [ ]:
def beam_frame(loader, label):
    """Extract per-frame (y, x) beam positions as a tidy DataFrame."""
    d = loader.data
    return pd.DataFrame({
        "fits_index": d["fits_index"].to_numpy(),
        "scan": d["scan"].to_numpy(),
        "sam_th": d["sam_th"].to_numpy(),
        f"y_{label}": [bs[0] for bs in d["beam_spot"]],
        f"x_{label}": [bs[1] for bs in d["beam_spot"]],
    })

cmp = beam_frame(simple, "simple").merge(
    beam_frame(current, "current")[["fits_index", "y_current", "x_current"]],
    on="fits_index",
)
cmp["dy"] = cmp["y_simple"] - cmp["y_current"]
cmp["dx"] = cmp["x_simple"] - cmp["x_current"]
cmp["dist"] = np.hypot(cmp["dy"], cmp["dx"])
cmp

### Frames where the trackers disagree by more than a few pixels

In [ ]:
DISAGREE_PX = 3
cmp[cmp["dist"] > DISAGREE_PX]

## 6. Beam trajectories vs frame

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, coord in zip(axes, ["x", "y"]):
    ax.plot(cmp["fits_index"], cmp[f"{coord}_simple"], "o-", ms=3, label="simple")
    ax.plot(cmp["fits_index"], cmp[f"{coord}_current"], "s--", ms=3, label="current")
    ax.set_xlabel("frame (fits_index)")
    ax.set_ylabel(f"beam {coord} [px, trimmed]")
    ax.legend()
axes[0].set_title("Beam column (x) vs frame")
axes[1].set_title("Beam row (y) vs frame")
fig.tight_layout()

## 7. Reflectivity: simple vs current

Overlays R vs q from each tracker (all energies/polarizations together).

In [ ]:
r_simple = simple.reduce()
r_current = current.reduce()

fig, ax = plt.subplots(figsize=(7, 5))
for label, r, style in [("simple", r_simple, "o-"), ("current", r_current, "s--")]:
    r = r.sort_values("q")
    ax.errorbar(r["q"], r["R"], yerr=r["R_err"], fmt=style, ms=3, lw=1,
                capsize=2, label=label)
ax.set_yscale("log")
ax.set_xlabel(r"q [$\AA^{-1}$]")
ax.set_ylabel("R [arb]")
ax.set_title("Reflectivity: simple vs current tracker")
ax.legend()
fig.tight_layout()

## 8. Inspect a single frame

Set `INSPECT_INDEX` to any `fits_index` (e.g. one flagged in section 5) to see
where each tracker placed the beam. Red **x** = simple, blue **+** = current.

In [ ]:
from matplotlib.colors import LogNorm

INSPECT_INDEX = int(simple.data["fits_index"].iloc[len(simple.data) // 2])

img = simple.get_clean_image(INSPECT_INDEX)  # trimmed+cleaned; same for both
row_s = simple.data.loc[simple.data["fits_index"] == INSPECT_INDEX]
row_c = current.data.loc[current.data["fits_index"] == INSPECT_INDEX]
bs_s = row_s["beam_spot"].iloc[0]
bs_c = row_c["beam_spot"].iloc[0]

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(np.clip(img, 1, None), norm=LogNorm(), cmap="terrain")
ax.plot(bs_s[1], bs_s[0], "rx", ms=12, mew=2, label=f"simple {tuple(bs_s)}")
ax.plot(bs_c[1], bs_c[0], "b+", ms=12, mew=2, label=f"current {tuple(bs_c)}")
ax.set_title(f"Frame {INSPECT_INDEX}  (sam_th={row_s['sam_th'].iloc[0]:.4f})")
ax.legend()
fig.tight_layout()

## 9. Stitch diagnostics

One row per detected stitch boundary: what **triggered** it (back-step and/or
which condition changed), the **conditions_changed** before→after values, how
many **overlap points** were used, and the fitted **scale**. A boundary with
`num_stitch_points == 0` or `failed == True` is a stitch that isn't working.
An empty table means no boundaries were detected — check that a condition
(hos/exposure/slits) actually changes and that `sam_th` steps back.

In [ ]:
# Uses whichever loader you processed above (the simple tracker here).
simple.diagnose_stitches()

## 10. (Optional) Save the simple-tracker result

Writes a clean, rounded `.dat` + plots (uses the export you already have).

In [ ]:
from pxr_reduce.dataset import ReducedDataset

# ReducedDataset.from_loader(simple).save("results/simple_tracked.dat")